# AOU-1 — Cohort definition. Phase M3 / Wave 1. Drives load_qc_cohort() from src/python/aou_ld_panel.py against the AoU v7 controlled-tier WGS MatrixTable. Emits 3 checkpointed MTs for Wave 2 dev fire.


**Init pattern (per `feedback_aou_dataproc_pyspark_submit_args` memory, baked 2026-05-12):** Cell 1a sets `PYSPARK_SUBMIT_ARGS` before any pyspark/hail import — this is the only lever that honors `spark.executor.cores=1` on AoU YARN. Cell 1b calls `hl.init()` directly (bypassing the `init_hail` wrapper whose `spark_conf=dict` path is silently dropped on YARN). Together, they pair with `naive_coalesce(2048)` in `aou_ld_panel.py:218` to clear the v8 partition-explosion RegionPool OOM (DEC-2026-05-04-01). Patch-verification at end of Cell 1b: prints checkpoint URIs from `_qc_checkpoint_uri` (commit 36e8062 / quick 260512-jd9) to confirm both the cores=1 lever AND the distinct sensitivity/primary checkpoint paths are live before any compute fires.

## ⚠ RUN PROTOCOL — one cohort at a time, smallest→largest, confirm between

**DO NOT "Run All".** Run cells top-to-bottom, but STOP after each cohort's validation cell, confirm it PASSED, and only then proceed to the next cohort. Cohorts run **smallest→largest** so the cheapest validates first and a failure costs the least:

1. **Cell 1a → 1a'' → 1b** — env guards + Hail init. Verify `executor.cores=1` and that `WORKSPACE_BUCKET = gs://rw-migration-aou-rw-476cdac2` (NOT a `cloned-mybucket` placeholder).
2. **Cell 3 → Cell 3.5** — AFR primary (~74k samples, smallest). Confirm 3.5 PASS, then continue.
3. **Cell 4 → Cell 4.5** — AFR sensitivity (~74k). Confirm 4.5 PASS, then continue.
4. **Cell 5 → Cell 5.5** — EUR parity (~221k, largest). Confirm 5.5 PASS, then continue.
5. **Cell 6** (disjoint-cohort sanity) → **Cell 7** (cohort-summary table).

Each cohort's genome-wide build runs the per-chromosome fan-out (chr1→chr22; watch chr1 clear its checkpoint in minutes = the wedge fix is working). **If a cohort fails, STOP — do not run the next cohort.**

In [ ]:
# Cell 1a — executor resources + requester-pays GCS billing at the spark-submit boundary.
# CANONICAL PATTERN (feedback_aou_dataproc_pyspark_submit_args, baked 2026-05-12): on AoU's
# Dataproc + YARN cluster, hl.init(spark_conf=dict) is silently overridden; PYSPARK_SUBMIT_ARGS
# injected BEFORE any pyspark/hail import IS honored (spark-submit boundary = highest precedence).
# This cell MUST run before any other pyspark/hail import. Pairs with naive_coalesce(2048) in
# aou_ld_panel.py (DEC-2026-05-04-01 v8 partition-explosion OOM remediation).
#
# REQUESTER-PAYS (added 2026-06-02): the controlled WGS bucket vwb-aou-datasets-controlled is
# requester-pays, and the migrated Verily Hail cluster does NOT pre-set the GCS-connector billing
# project (classic AoU did, in spark-defaults). CUSTOM mode scopes the userProject billing header
# to ONLY that bucket (the non-RP output bucket is untouched). GOTCHAS: bucket NAME only, NO gs://
# prefix (prefix => silent match failure); project = $GOOGLE_PROJECT; must be set pre-SparkContext.
# If reads still 400 "requester pays ... no user project", the env forced the newer STORAGE_CLIENT
# => add: --conf spark.hadoop.fs.gs.client.type=HTTP_API_CLIENT (RP honored only on HTTP_API_CLIENT).
import os
_proj = os.environ.get("GOOGLE_PROJECT", "wb-perky-corn-6639")
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--conf spark.executor.cores=1 "
    "--conf spark.executor.memory=5g "
    "--conf spark.driver.cores=1 "
    "--conf spark.hadoop.fs.gs.requester.pays.mode=CUSTOM "
    "--conf spark.hadoop.fs.gs.requester.pays.buckets=vwb-aou-datasets-controlled "
    f"--conf spark.hadoop.fs.gs.requester.pays.project.id={_proj} "
    "pyspark-shell"
)
print("PYSPARK_SUBMIT_ARGS set:", os.environ["PYSPARK_SUBMIT_ARGS"])

In [ ]:
# Cell 1a'' — env pins (HARD override; defeats saved-template 404 placeholder pollution).
# The migrated Verily Hail Dataproc cluster does NOT auto-set WORKSPACE_BUCKET or the WGS input
# path (classic AoU did; only GOOGLE_PROJECT is auto-set). A saved/duplicated cluster template can
# inject a 404 placeholder (gs://cloned-mybucket-<project>) into WORKSPACE_BUCKET, so setdefault is
# UNSAFE — hard-assign the verified migrated values. See feedback_aou_cluster_template_bucket_pollution.
import os
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-476cdac2"
os.environ["WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH"] = (
    "gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/multiMT/hail.mt"
)
print("WORKSPACE_BUCKET =", os.environ["WORKSPACE_BUCKET"])
print("WGS path         =", os.environ["WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH"])
print("GOOGLE_PROJECT   =", os.environ.get("GOOGLE_PROJECT", "(unset!)"))

In [ ]:
# Cell 1b — Initialize Hail with spark_conf threading + verify executor.cores=1.
# Calls hl.init directly (NOT through init_hail wrapper) because the wrapper's
# spark_conf path is known broken on AoU YARN; the PYSPARK_SUBMIT_ARGS lever
# from Cell 1a is what actually binds the conf. The spark_conf dict here is
# belt-and-suspenders for portability — preserves the conf-by-dict path for
# environments where it works (local Spark, standalone clusters), while AoU
# binds via the env-var lever.
import sys
import os
# Portable sys.path (HOME-relative): migrated Verily Hail Dataproc runs as
# user 'dataproc' (HOME=/home/dataproc), not 'jupyter'. expanduser('~') resolves
# regardless of cluster home dir, given the repo was cloned to ~/coloc_analysis.
sys.path.insert(0, os.path.expanduser("~/coloc_analysis/src/python"))
import hail as hl
hl.init(
    default_reference="GRCh38",
    log="/tmp/hail.log",
    quiet=True,
    spark_conf={
        "spark.executor.cores": "1",
        "spark.executor.memory": "5g",
        "spark.driver.cores": "1",
    },
)
# Pull cohort helpers AFTER Hail backend is up (so any aou_ld_panel-side
# Hail-dependent imports succeed):
from aou_ld_panel import load_qc_cohort, ANCESTRY_FIELD, KING_KINSHIP_THRESHOLD, _qc_checkpoint_uri

# Verify the patches are live (cores=1 confirms PYSPARK_SUBMIT_ARGS bound;
# _qc_checkpoint_uri import confirms commit 36e8062 is in the AoU clone):
sc_conf = hl.spark_context().getConf()
cores = sc_conf.get('spark.executor.cores')
assert cores == '1', (
    f"PYSPARK_SUBMIT_ARGS lever did not bind — got cores={cores}, expected '1'. "
    f"DO NOT proceed to Cell 3+ — the v8 partition-explosion OOM config is NOT live. "
    f"Action: Kernel menu → Restart Kernel; then re-fire Cell 1a + Cell 1b."
)
print("=== HAIL INIT ===")
print(f"  Hail version          : {hl.__version__}")
print(f"  spark.executor.cores  : {cores}  OK")
print(f"  spark.executor.memory : {sc_conf.get('spark.executor.memory')}")
print(f"  spark.driver.cores    : {sc_conf.get('spark.driver.cores')}")
print(f"  spark.master          : {sc_conf.get('spark.master')}")
print()
print("=== ENV ===")
print(f"  WORKSPACE_BUCKET = {os.environ['WORKSPACE_BUCKET']}")
print(f"  GOOGLE_PROJECT   = {os.environ['GOOGLE_PROJECT']}")
print(f"  WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH = {os.environ['WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH']}")
print()
print("=== PATCH VERIFICATION (commit 36e8062 — m3-W1-checkpoint-suffix; quick 260512-jd9) ===")
print(f"  AFR primary URI     : {_qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'afr', False)}")
print(f"  AFR sensitivity URI : {_qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'afr', True)}")
print(f"  EUR parity URI      : {_qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'eur', False)}")

In [ ]:
# Cell 3 — Primary AFR cohort (D-M3-07 PCA-primary)
mt_afr = load_qc_cohort(
    mt_path=os.environ["WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH"],
    ancestry="afr",
    sensitivity=False,
)
n_afr = mt_afr.count_cols()
n_var_afr = mt_afr.count_rows()
print(f"AFR PCA cohort: {n_afr} samples, {n_var_afr} variants")
# Already checkpointed to gs://${WORKSPACE_BUCKET}/ld/mt_afr_qc.mt by load_qc_cohort()

In [ ]:
# Cell 3.5 — Mandatory post-write bucket-contents validation.
# m3-W1-empty-mt-catastrophe (2026-05-21) regression guard.
#
# Hail's mt.checkpoint() writes _SUCCESS on driver-side
# tasks-reported-complete accounting WITHOUT validating output
# contents. Under spark.executor.cores=1/mem=5g, executors can
# silently truncate after writing Parquet schema footers — leaving
# the catastrophe MT-skeleton (_SUCCESS + 35-byte rows-stubs +
# absent entries/rows/parts/). The post-write count_rows/
# count_cols assertions inside load_qc_cohort (commit chain ending
# in m3-W1 Track 4 patch 4/7) are the first line of defense; this
# cell is the second line — operates on bucket state directly,
# not on Hail-internal in-memory state.
#
# DO NOT proceed to the next cell if this assertion fails.
# Pause environment, preserve hail.log to bucket, investigate.
#
# Cross-references:
# - .planning/debug/m3-W1-empty-mt-catastrophe.md
# - [[feedback_aou_success_marker_not_evidence_of_data]]
# - [[feedback_hail_checkpoint_contract_violation]]
#
# Cohort: Primary AFR (D-M3-07 PCA-primary), validates mt_afr_qc.mt
import subprocess
_ckpt_uri = _qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'afr', False)
_entries_dir = _ckpt_uri.rstrip('/') + '/entries/rows/parts/'
_r = subprocess.run(
    ['gsutil', 'du', '-s', _entries_dir],
    capture_output=True, text=True,
)
assert _r.returncode == 0, (
    f'bucket inspection failed at {_entries_dir}: returncode={_r.returncode}; '
    f'stderr={_r.stderr.strip()}. Likely entries/ directory absent — '
    f'the m3-W1 empty-MT catastrophe signature.'
)
_size_bytes = int(_r.stdout.split()[0])
_MIN_BYTES = 1000000000  # 1.0 GB floor for production fires (chr22 smoke runs need a lower-threshold variant)
assert _size_bytes > _MIN_BYTES, (
    f'MT entries at {_entries_dir} is {_size_bytes:,} bytes '
    f'(< {_MIN_BYTES:,} bytes floor) — empty-MT catastrophe regression '
    f'guard. Cohort write produced a stub MT; do NOT proceed. '
    f'See .planning/debug/m3-W1-empty-mt-catastrophe.md.'
)
print(f'OK: {_ckpt_uri} populated ({_size_bytes / 10**9:.2f} GB at entries/rows/parts/)')


In [ ]:
# Cell 4 — AFR sensitivity cohort (D-M3-07 self-report Black/African American sensitivity)
mt_afr_selfid = load_qc_cohort(
    mt_path=os.environ["WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH"],
    ancestry="afr",
    sensitivity=True,
)
n_afr_selfid = mt_afr_selfid.count_cols()
print(f"AFR PCA + self-id Black/AA cohort: {n_afr_selfid} samples (subset of AFR PCA cohort)")
# Checkpoint at gs://${WORKSPACE_BUCKET}/ld/mt_afr_pca_selfid_qc.mt

In [ ]:
# Cell 4.5 — Mandatory post-write bucket-contents validation.
# m3-W1-empty-mt-catastrophe (2026-05-21) regression guard.
#
# Hail's mt.checkpoint() writes _SUCCESS on driver-side
# tasks-reported-complete accounting WITHOUT validating output
# contents. Under spark.executor.cores=1/mem=5g, executors can
# silently truncate after writing Parquet schema footers — leaving
# the catastrophe MT-skeleton (_SUCCESS + 35-byte rows-stubs +
# absent entries/rows/parts/). The post-write count_rows/
# count_cols assertions inside load_qc_cohort (commit chain ending
# in m3-W1 Track 4 patch 4/7) are the first line of defense; this
# cell is the second line — operates on bucket state directly,
# not on Hail-internal in-memory state.
#
# DO NOT proceed to the next cell if this assertion fails.
# Pause environment, preserve hail.log to bucket, investigate.
#
# Cross-references:
# - .planning/debug/m3-W1-empty-mt-catastrophe.md
# - [[feedback_aou_success_marker_not_evidence_of_data]]
# - [[feedback_hail_checkpoint_contract_violation]]
#
# Cohort: AFR sensitivity (D-M3-07 self-report), validates mt_afr_pca_selfid_qc.mt
import subprocess
_ckpt_uri = _qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'afr', True)
_entries_dir = _ckpt_uri.rstrip('/') + '/entries/rows/parts/'
_r = subprocess.run(
    ['gsutil', 'du', '-s', _entries_dir],
    capture_output=True, text=True,
)
assert _r.returncode == 0, (
    f'bucket inspection failed at {_entries_dir}: returncode={_r.returncode}; '
    f'stderr={_r.stderr.strip()}. Likely entries/ directory absent — '
    f'the m3-W1 empty-MT catastrophe signature.'
)
_size_bytes = int(_r.stdout.split()[0])
_MIN_BYTES = 1000000000  # 1.0 GB floor for production fires (chr22 smoke runs need a lower-threshold variant)
assert _size_bytes > _MIN_BYTES, (
    f'MT entries at {_entries_dir} is {_size_bytes:,} bytes '
    f'(< {_MIN_BYTES:,} bytes floor) — empty-MT catastrophe regression '
    f'guard. Cohort write produced a stub MT; do NOT proceed. '
    f'See .planning/debug/m3-W1-empty-mt-catastrophe.md.'
)
print(f'OK: {_ckpt_uri} populated ({_size_bytes / 10**9:.2f} GB at entries/rows/parts/)')


In [ ]:
# Cell 5 — EUR parity cohort (D-M3-01)
mt_eur = load_qc_cohort(
    mt_path=os.environ["WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH"],
    ancestry="eur",
    sensitivity=False,
)
n_eur = mt_eur.count_cols()
print(f"EUR PCA cohort: {n_eur} samples")
# Checkpoint at gs://${WORKSPACE_BUCKET}/ld/mt_eur_qc.mt

In [ ]:
# Cell 5.5 — Mandatory post-write bucket-contents validation.
# m3-W1-empty-mt-catastrophe (2026-05-21) regression guard.
#
# Hail's mt.checkpoint() writes _SUCCESS on driver-side
# tasks-reported-complete accounting WITHOUT validating output
# contents. Under spark.executor.cores=1/mem=5g, executors can
# silently truncate after writing Parquet schema footers — leaving
# the catastrophe MT-skeleton (_SUCCESS + 35-byte rows-stubs +
# absent entries/rows/parts/). The post-write count_rows/
# count_cols assertions inside load_qc_cohort (commit chain ending
# in m3-W1 Track 4 patch 4/7) are the first line of defense; this
# cell is the second line — operates on bucket state directly,
# not on Hail-internal in-memory state.
#
# DO NOT proceed to the next cell if this assertion fails.
# Pause environment, preserve hail.log to bucket, investigate.
#
# Cross-references:
# - .planning/debug/m3-W1-empty-mt-catastrophe.md
# - [[feedback_aou_success_marker_not_evidence_of_data]]
# - [[feedback_hail_checkpoint_contract_violation]]
#
# Cohort: EUR parity (D-M3-01), validates mt_eur_qc.mt
import subprocess
_ckpt_uri = _qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'eur', False)
_entries_dir = _ckpt_uri.rstrip('/') + '/entries/rows/parts/'
_r = subprocess.run(
    ['gsutil', 'du', '-s', _entries_dir],
    capture_output=True, text=True,
)
assert _r.returncode == 0, (
    f'bucket inspection failed at {_entries_dir}: returncode={_r.returncode}; '
    f'stderr={_r.stderr.strip()}. Likely entries/ directory absent — '
    f'the m3-W1 empty-MT catastrophe signature.'
)
_size_bytes = int(_r.stdout.split()[0])
_MIN_BYTES = 1000000000  # 1.0 GB floor for production fires (chr22 smoke runs need a lower-threshold variant)
assert _size_bytes > _MIN_BYTES, (
    f'MT entries at {_entries_dir} is {_size_bytes:,} bytes '
    f'(< {_MIN_BYTES:,} bytes floor) — empty-MT catastrophe regression '
    f'guard. Cohort write produced a stub MT; do NOT proceed. '
    f'See .planning/debug/m3-W1-empty-mt-catastrophe.md.'
)
print(f'OK: {_ckpt_uri} populated ({_size_bytes / 10**9:.2f} GB at entries/rows/parts/)')


In [ ]:
# Cell 6 — Disjoint-cohort sanity check (RESEARCH O5)
afr_samples = mt_afr.s.collect()
eur_samples = mt_eur.s.collect()
overlap = set(afr_samples) & set(eur_samples)
assert len(overlap) == 0, f"AFR and EUR cohorts overlap by {len(overlap)} samples; investigate!"
print(f"OK: AFR and EUR cohorts disjoint ({len(afr_samples)} + {len(eur_samples)} samples)")

In [ ]:
# Cell 7 — Cohort-summary table for the validation memo
import pandas as pd
cohort_summary = pd.DataFrame({
    "cohort": ["AFR_pca", "AFR_pca_selfid", "EUR_pca"],
    "n_samples": [n_afr, n_afr_selfid, n_eur],
    "n_variants": [n_var_afr, mt_afr_selfid.count_rows(), mt_eur.count_rows()],
    "kinship_threshold": [KING_KINSHIP_THRESHOLD] * 3,
    "ancestry_field": [ANCESTRY_FIELD] * 3,
    "checkpoint_path": [
        _qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'afr', False),
        _qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'afr', True),
        _qc_checkpoint_uri(os.environ['WORKSPACE_BUCKET'], 'eur', False),
    ],
})
cohort_summary.to_csv("cohort_summary_m3.tsv", sep="\t", index=False)
print(cohort_summary)

## Output: 3 checkpointed MTs in workspace bucket. Mirror cohort_summary_m3.tsv to NCSU GPFS at .planning/phases/m3-aou-afr-ld-panel-build/cohort_summary_m3.tsv after Wave 2 dev fire signoff (Wave 5 close-out task).